# FD001 Boosting Models

Objetivo: comparar la baseline `temporal_w30 + RandomForestRegressor` contra modelos tabulares fuertes usando solo validación artificial por motores y cutoffs. No se usa el test oficial FD001.


In [1]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
CUT_RULS = (20, 50, 80, 110, 140)
DEFAULT_WINDOW_SIZE = 30
DEFAULT_RUL_CAP = 125

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [2]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    plot_validation_diagnostics,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)


def add_bin_metric_columns(metrics, bin_metrics):
    result = metrics.copy()
    for label in ["0-30", "30-60", "60-90", "90+"]:
        safe_label = label.replace("-", "_").replace("+", "plus")
        subset = bin_metrics.loc[bin_metrics["rul_bin"].astype(str) == label]
        for _, row in subset.iterrows():
            mask = (
                (result["representation"] == row["representation"])
                & (result["model_name"] == row["model_name"])
            )
            result.loc[mask, f"mae_rul_{safe_label}"] = row["mae"]
            result.loc[mask, f"dangerous_error_pct_rul_{safe_label}"] = row["dangerous_error_pct"]
    return result


def available_tabular_factories(random_state=42):
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )

    factories = OrderedDict()
    availability = []
    has_external_boosting = False

    factories["RandomForestRegressor"] = lambda: RandomForestRegressor(
        n_estimators=250,
        max_depth=14,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1,
    )

    try:
        from xgboost import XGBRegressor
        factories["XGBRegressor"] = lambda: XGBRegressor(
            n_estimators=160,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )
        has_external_boosting = True
        availability.append("XGBoost disponible: se incluye XGBRegressor.")
    except Exception as exc:
        availability.append(f"XGBoost no disponible: {type(exc).__name__}.")

    try:
        from lightgbm import LGBMRegressor
        factories["LGBMRegressor"] = lambda: LGBMRegressor(
            n_estimators=220,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=0.1,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        has_external_boosting = True
        availability.append("LightGBM disponible: se incluye LGBMRegressor.")
    except Exception as exc:
        availability.append(f"LightGBM no disponible: {type(exc).__name__}.")

    if not has_external_boosting:
        availability.append("Sin XGBoost/LightGBM: se usan fallbacks sklearn.")
        factories["HistGradientBoostingRegressor"] = lambda: HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=random_state,
        )
        factories["GradientBoostingRegressor"] = lambda: GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.85,
            random_state=random_state,
        )
        factories["ExtraTreesRegressor"] = lambda: ExtraTreesRegressor(
            n_estimators=220,
            max_depth=16,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    return factories, availability

def fit_predict_model(prepared, model_name, model, representation, sample_weight=None):
    if sample_weight is None:
        model.fit(prepared["X_train"], prepared["y_train"])
    else:
        model.fit(prepared["X_train"], prepared["y_train"], sample_weight=sample_weight)
    preds = model.predict(prepared["X_eval"])
    return prediction_frame(
        prepared["eval_df"],
        preds,
        model_name=model_name,
        representation=representation,
    ), model


## Preparación

Se usan features temporales tabulares resumidas, no ventanas crudas como secuencia. El scaler se ajusta solo con los motores de train.


In [3]:
set_global_seed(SEED)
prepared = prepare_fd001_temporal_validation_only(
    data_dir=DATA_DIR,
    eval_size=EVAL_SIZE,
    random_state=SEED,
    max_rul=DEFAULT_RUL_CAP,
    cut_ruls=CUT_RULS,
    window_size=DEFAULT_WINDOW_SIZE,
)
preprocessing_summary(prepared)


,split,rows,units,features,target_mean,target_min,target_max
0,train,16561,80,119,86.958819,0.0,125.0
1,eval,99,20,119,79.393939,20.0,140.0


## Modelos

Si XGBoost o LightGBM no están instalados, el notebook sigue con fallbacks de sklearn.


In [4]:
model_factories, availability_notes = available_tabular_factories(random_state=SEED)
print("\n".join(availability_notes))
list(model_factories.keys())


XGBoost disponible: se incluye XGBRegressor.
LightGBM disponible: se incluye LGBMRegressor.


['RandomForestRegressor', 'XGBRegressor', 'LGBMRegressor']

## Entrenamiento y validación artificial


In [5]:
representation = f"temporal_w{DEFAULT_WINDOW_SIZE}"
prediction_tables = []
fitted_models = {}

for model_name, factory in model_factories.items():
    print(f"Entrenando {model_name}...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pred_df, model = fit_predict_model(prepared, model_name, factory(), representation)
    prediction_tables.append(pred_df)
    fitted_models[model_name] = model

predictions = pd.concat(prediction_tables, ignore_index=True)
metrics = metrics_by_model(predictions)
metrics.insert(2, "window_size", DEFAULT_WINDOW_SIZE)
metrics.insert(3, "rul_cap", DEFAULT_RUL_CAP)
metrics.insert(4, "n_features", len(prepared["feature_columns"]))
metrics.insert(5, "target_used_for_training", "RUL_capped")

bin_metrics = metrics_by_rul_bin(predictions)
metrics = add_bin_metric_columns(metrics, bin_metrics)

metrics


Entrenando RandomForestRegressor...
Entrenando XGBRegressor...
Entrenando LGBMRegressor...


,representation,model_name,window_size,rul_cap,n_features,target_used_for_training,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus
0,temporal_w30,LGBMRegressor,30,125,119,RUL_capped,99,12.809901,16.517134,0.846858,397.315013,4.013283,11.111111,4.325694,0.0,8.545874,15.0,16.484272,40.0,17.463165,0.0
1,temporal_w30,XGBRegressor,30,125,119,RUL_capped,99,13.931649,17.238230,0.833194,418.575570,4.228036,9.090909,4.847880,0.0,12.338805,10.0,17.969041,35.0,17.336378,0.0
2,temporal_w30,RandomForestRegressor,30,125,119,RUL_capped,99,13.817341,17.912073,0.819898,580.933845,5.868019,14.141414,3.683619,0.0,12.365219,20.0,19.480671,50.0,16.854527,0.0


## Guardado


In [6]:
figure_dir = FIGURES_DIR / "fd001_boosting"
figure_dir.mkdir(parents=True, exist_ok=True)

metrics.to_csv(RESULTS_DIR / "fd001_boosting_metrics.csv", index=False)
predictions.to_csv(RESULTS_DIR / "fd001_boosting_predictions.csv", index=False)
bin_metrics.to_csv(RESULTS_DIR / "fd001_boosting_metrics_by_rul_bin.csv", index=False)
plot_validation_diagnostics(predictions, figure_dir, "FD001 boosting validation")

print("Guardado:")
print(RESULTS_DIR / "fd001_boosting_metrics.csv")
print(RESULTS_DIR / "fd001_boosting_predictions.csv")
print(figure_dir)


Guardado:
C:\Users\lauta\OneDrive\Escritorio\ML\TPF-ML\results\fd001_boosting_metrics.csv
C:\Users\lauta\OneDrive\Escritorio\ML\TPF-ML\results\fd001_boosting_predictions.csv
C:\Users\lauta\OneDrive\Escritorio\ML\TPF-ML\figures\fd001_boosting


## Lectura rápida


In [7]:
display(metrics.sort_values(["rmse", "mae"]))
display(bin_metrics.sort_values(["rul_bin", "rmse"]))


,representation,model_name,window_size,rul_cap,n_features,target_used_for_training,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus
0,temporal_w30,LGBMRegressor,30,125,119,RUL_capped,99,12.809901,16.517134,0.846858,397.315013,4.013283,11.111111,4.325694,0.0,8.545874,15.0,16.484272,40.0,17.463165,0.0
1,temporal_w30,XGBRegressor,30,125,119,RUL_capped,99,13.931649,17.238230,0.833194,418.575570,4.228036,9.090909,4.847880,0.0,12.338805,10.0,17.969041,35.0,17.336378,0.0
2,temporal_w30,RandomForestRegressor,30,125,119,RUL_capped,99,13.817341,17.912073,0.819898,580.933845,5.868019,14.141414,3.683619,0.0,12.365219,20.0,19.480671,50.0,16.854527,0.0


,representation,model_name,rul_bin,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct
3,temporal_w30,RandomForestRegressor,0-30,20,3.683619,4.183181,0.000000,8.942007,0.447100,0.0
11,temporal_w30,LGBMRegressor,0-30,20,4.325694,5.371490,0.000000,11.705092,0.585255,0.0
7,temporal_w30,XGBRegressor,0-30,20,4.847880,6.082971,0.000000,12.614984,0.630749,0.0
10,temporal_w30,LGBMRegressor,30-60,20,8.545874,11.063005,0.000000,39.388014,1.969401,15.0
6,temporal_w30,XGBRegressor,30-60,20,12.338805,14.204751,0.000000,65.815416,3.290771,10.0
2,temporal_w30,RandomForestRegressor,30-60,20,12.365219,16.912432,0.000000,174.620796,8.731040,20.0
9,temporal_w30,LGBMRegressor,60-90,20,16.484272,19.676366,0.000000,170.049283,8.502464,40.0
5,temporal_w30,XGBRegressor,60-90,20,17.969041,20.030033,0.000000,151.382227,7.569111,35.0
1,temporal_w30,RandomForestRegressor,60-90,20,19.480671,22.727859,0.000000,231.908889,11.595444,50.0
0,temporal_w30,RandomForestRegressor,90+,39,16.854527,19.846653,-0.751772,165.462153,4.242619,0.0
